# Disagreement GEC v2: the LLM answers only the blanks, the code rebuilds the line (Gemma-3-12B)

Same idea as `disagreement_few_shot_gemma.ipynb`: the three transcripts are aligned word by word (as in ROVER), words all three agree on are kept, and each place where they disagree becomes a numbered blank listing what each recogniser wrote.

**What changed from v1, and why.** In v1 the LLM had to write the whole line back with the blanks filled. On long clips Gemma often wrote only its answers to the blanks and left out the agreed words between them (for example, 118 words in, 25 words out). Deletions rose to 16.6% on the UrduSpeech corpus. In v2:

1. The LLM outputs only one numbered answer per blank (`1: words`, `2: words`, ...). Prompt `disagreement_fill_few_shot` in `PROMPTS.md`.
2. The notebook puts the agreed words and the answers back together, so the agreed words can never be dropped or changed.
3. Any blank the LLM does not answer (missing number, cut-off reply, unreadable line) gets the ROVER vote for that blank, so a failed answer falls back to ROVER instead of losing words.
4. The few-shot examples show the reference words for each blank, found by aligning the reference to the same word network.
5. The raw reply, the number of blanks and the number answered are saved for every clip.

Clips where the three transcripts fully agree make no LLM call. The result is scored against each of the three baselines and against ROVER (experiment 04).


## 1. Setup

In [ ]:
FOLDER = "05_disagreement_gec"
NAME = "disagreement_few_shot_gemma_v2"
LIMIT = None
WORKERS = 8
CHUNK = 200
MAX_TOKENS = 1024

import os, re, json, time, string, unicodedata, platform, importlib.metadata
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, jiwer

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA = ROOT / "data"
RESULTS = ROOT / "experiments" / FOLDER / "results"
for sub in ("corrections", "scores", "run_info"):
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)

RECOGNISERS = {
    "whisper_large_v3": "Whisper-large-v3",
    "omnilingual_llm_1b_v2": "Omnilingual LLM 1B v2",
    "indic_transcribe_core": "Indic Transcribe Core",
}
SHORT = {"whisper_large_v3": "Whisper", "omnilingual_llm_1b_v2": "Omni", "indic_transcribe_core": "Indic"}

# clean_urdu v1
_DIAC_RE = re.compile("[ؐ-ؚـً-ٰۖ-ۜ۟-۪ۨ-ۭ]")
_PUNCT = set(string.punctuation) | set("۔،؟؛٪।॥“”"
                                       "‘’—–…«»‖‐‒·")
_IZAFAT = {"اے", "ای", "ئے", "ء", "ے", "او"}
_UMAP = {"ي": "ی", "ك": "ک", "ة": "ہ", "ۃ": "ہ",
         "ى": "ی", "إ": "ا", "أ": "ا", "آ": "ا",
         "ٲ": "ا", "ٳ": "ا", "ؤ": "و", "ئ": "ی"}

def clean_urdu(text):
    t = unicodedata.normalize("NFC", str(text))
    t = _DIAC_RE.sub("", t)
    t = "".join(_UMAP.get(ch, ch) for ch in t)
    t = "".join(" " if ch in _PUNCT else ch for ch in t)
    return " ".join(w for w in t.split() if w not in _IZAFAT).strip()


def prompt_text(text):
    t = _DIAC_RE.sub("", unicodedata.normalize("NFC", str(text)))
    t = "".join(" " if ch in _PUNCT else ch for ch in t)
    return " ".join(t.split())


by_clip = pd.read_csv(DATA / "transcripts" / "all_recognisers_by_clip.csv", keep_default_na=False)
assert len(by_clip) == 2400, len(by_clip)
work_clips = by_clip.sample(LIMIT, random_state=42).sort_index() if LIMIT else by_clip
print(f"{len(by_clip)} clips; this run covers {len(work_clips)}")

examples = pd.read_csv(DATA / "few_shot_examples.csv", keep_default_na=False)
assert len(examples) == 5
assert not set(examples.clip_id) & set(by_clip.clip_id), "example clips must not be scored clips"
print("Few-shot example clips:", ", ".join(examples.clip_id))

## 2. LLM

In [ ]:
import requests

LLAMA_URL = "http://127.0.0.1:8080"
LLM_NAME = "Gemma-3-12B (llama.cpp server)"


def call_llm(prompt, temperature=0.0, max_tokens=512):
    last = None
    for attempt in range(5):
        try:
            r = requests.post(f"{LLAMA_URL}/v1/chat/completions", timeout=600, json={
                "model": "gemma", "messages": [{"role": "user", "content": prompt}],
                "temperature": temperature, "max_tokens": max_tokens})
            r.raise_for_status()
            j = r.json()
            choice = j["choices"][0]
            return {"text": choice["message"]["content"] or "",
                    "tokens_in": j["usage"]["prompt_tokens"], "tokens_out": j["usage"]["completion_tokens"],
                    "cut": choice.get("finish_reason") == "length"}
        except Exception as e:
            last = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"{type(last).__name__}: {str(last)[:150]}")

## 3. Blanks, prompt and answers

In [ ]:
def run_jobs(jobs, work):
    path = RESULTS / "corrections" / f"{NAME}.csv"
    done = set()
    if path.exists():
        d = pd.read_csv(path, keep_default_na=False)
        d = d[~d.status.astype(str).str.startswith("error")]
        done = set(zip(d.clip_id, d.condition))
    todo = [j for j in jobs if (j["clip_id"], j["condition"]) not in done]
    print(f"{len(done)} outputs already saved, {len(todo)} to go")
    wall = 0.0
    for s in range(0, len(todo), CHUNK):
        chunk = todo[s:s + CHUNK]
        t0 = time.perf_counter()
        with ThreadPoolExecutor(WORKERS) as pool:
            results = list(pool.map(work, chunk))
        wall += time.perf_counter() - t0
        failed = sum(str(r["status"]).startswith("error") for r in results)
        if failed > len(chunk) // 2:
            raise RuntimeError(f"{failed} of {len(chunk)} calls failed ({results[0]['status']}). This chunk was not saved. "
                               "Fix the LLM, then run this notebook again: finished clips are skipped.")
        rows = [{"clip_id": j["clip_id"], "condition": j["condition"], **r} for j, r in zip(chunk, results)]
        pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False, encoding="utf-8")
        print(f"  {min(s + CHUNK, len(todo))}/{len(todo)} done, {failed} errors in this chunk, {wall / 60:.1f} min")
    return wall

ORDER = ["indic_transcribe_core", "omnilingual_llm_1b_v2", "whisper_large_v3"]
VIEW_ORDER = ["whisper_large_v3", "omnilingual_llm_1b_v2", "indic_transcribe_core"]
NOTHING = "(nothing)"


def add_system(network, words, k):
    n, m = len(network), len(words)
    slots = [{w for w in slot if w is not None} for slot in network]
    cost = np.zeros((n + 1, m + 1), dtype=int)
    cost[:, 0] = np.arange(n + 1)
    cost[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sub = cost[i - 1, j - 1] + (0 if words[j - 1] in slots[i - 1] else 1)
            cost[i, j] = min(sub, cost[i - 1, j] + 1, cost[i, j - 1] + 1)
    i, j, out = n, m, []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and cost[i, j] == cost[i - 1, j - 1] + (0 if words[j - 1] in slots[i - 1] else 1):
            out.append(network[i - 1] + [words[j - 1]])
            i, j = i - 1, j - 1
        elif i > 0 and cost[i, j] == cost[i - 1, j] + 1:
            out.append(network[i - 1] + [None])
            i -= 1
        else:
            out.append([None] * k + [words[j - 1]])
            j -= 1
    return out[::-1]


def build_network(transcripts):
    network = [[w] for w in transcripts[ORDER[0]].split()]
    for k, stem in enumerate(ORDER[1:], 1):
        network = add_system(network, transcripts[stem].split(), k)
    return network


def vote(slot):
    counts = {}
    for idx, w in enumerate(slot):
        counts.setdefault(w, [0, idx])[0] += 1
    return max(counts.items(), key=lambda kv: (kv[1][0], -kv[1][1]))[0]


def template(transcripts, reference=None):
    """Split the aligned network into fixed words and blanks.

    Returns a list of parts: a string for a word all three agree on, or a dict for a blank with
    each recogniser's words, the ROVER vote for the blank and (for examples) the reference words."""
    network = build_network(transcripts)
    if reference is not None:
        network = add_system(network, reference.split(), 3)
    parts, run = [], []

    def flush():
        if run:
            blank = {"options": {s: " ".join(c[ORDER.index(s)] for c in run if c[ORDER.index(s)] is not None) or NOTHING
                                 for s in VIEW_ORDER},
                     "vote": " ".join(w for w in (vote(c[:3]) for c in run if any(x is not None for x in c[:3])) if w is not None)}
            if reference is not None:
                blank["reference"] = " ".join(c[3] for c in run if c[3] is not None) or NOTHING
            parts.append(blank)
            run.clear()

    for col in network:
        systems = col[:3]
        if all(x is None for x in systems):
            # a reference word that no recogniser wrote: it belongs to the neighbouring blank if there is one
            if run:
                run.append(col)
            elif parts and isinstance(parts[-1], dict) and reference is not None:
                parts[-1]["reference"] = (parts[-1]["reference"].replace(NOTHING, "") + " " + col[3]).strip()
            continue
        if len(set(systems)) == 1:
            flush()
            parts.append(systems[0])
        else:
            run.append(col)
    flush()
    return parts


def blanks_of(parts):
    return [p for p in parts if isinstance(p, dict)]


def view_block(parts):
    shown, n = [], 0
    for p in parts:
        if isinstance(p, dict):
            n += 1
            shown.append(f"[{n}]")
        else:
            shown.append(p)
    lines = [f"Blank {i} - " + " | ".join(f"{SHORT[s]}: {b['options'][s]}" for s in VIEW_ORDER)
             for i, b in enumerate(blanks_of(parts), 1)]
    return f"Line: {' '.join(shown)}\n" + "\n".join(lines)


def assemble(parts, fills):
    words, n = [], 0
    for p in parts:
        if isinstance(p, dict):
            n += 1
            words.append(fills.get(n, p["vote"]))
        else:
            words.append(p)
    return " ".join(" ".join(words).split())


_ANSWER = re.compile(r"^\s*\[?\s*(\d+)\s*\]?\s*[:：\-.)]\s*(.*)$")
_LABEL = re.compile(r"^(Whisper|Omni|Indic)\s*:\s*", re.I)


def parse_fills(text, n_blanks):
    fills = {}
    for line in str(text).splitlines():
        m = _ANSWER.match(line)
        if not m:
            continue
        i, value = int(m.group(1)), m.group(2).strip()
        if not 1 <= i <= n_blanks or i in fills or "|" in value:
            continue
        value = _LABEL.sub("", value).strip().strip('"\'')
        if value.lower().strip("() ") in ("nothing", "empty", ""):
            value = ""
        fills[i] = prompt_text(value)
    return fills


HEAD = """You are a strict Urdu ASR post-editor. Three speech recognisers transcribed the same audio and disagreed in some places. The words they agree on are kept. Each place where they disagree is a numbered blank in square brackets, with the options the recognisers wrote.
Rules:
- Fill every blank. You may pick one of its options, or write a different word or phrase if none of them fits. An option "(nothing)" means a recogniser wrote nothing there; answer (nothing) to leave the blank empty.
- Use the words around the blank to choose what makes sense in the sentence.
- Do NOT translate.
- Do NOT add punctuation, diacritics or any explanation.
Output ONLY one line per blank, in order, in the form "number: words". Do not repeat the line."""


def answer_block(parts):
    return "\n".join(f"{i}: {b['reference']}" for i, b in enumerate(blanks_of(parts), 1))


def transcripts_of(r):
    return {stem: prompt_text(getattr(r, stem)) for stem in RECOGNISERS}


example_parts = [template(transcripts_of(r), prompt_text(r.reference)) for r in examples.itertuples()]
EXAMPLE_BLOCK = "Here are 5 examples:\n\n" + "\n\n".join(
    f"{view_block(p)}\nAnswer:\n{answer_block(p)}" for p in example_parts)
print(EXAMPLE_BLOCK[:1500])

CONDITIONS = {"disagreement_v2": {"label": "Gemma few-shot disagreement v2", "inputs": list(RECOGNISERS)}}

jobs = []
for r in work_clips.itertuples():
    parts = template(transcripts_of(r))
    jobs.append({"clip_id": r.clip_id, "condition": "disagreement_v2", "parts": parts,
                 "prompt": f"{HEAD}\n\n{EXAMPLE_BLOCK}\n\nNow fill the blanks of this line. Output only the numbered answers.\n{view_block(parts)}\nAnswer:"})
print(f"{sum(bool(blanks_of(j['parts'])) for j in jobs)} of {len(jobs)} clips have at least one disagreement")


def work(job):
    parts = job["parts"]
    n = len(blanks_of(parts))
    rover = assemble(parts, {})
    base = {"blanks": n, "blanks_filled": 0, "raw": ""}
    if n == 0:
        return {"output": rover, "status": "no_disagreement", "llm_seconds": 0, "calls": 0, "tokens_in": 0, "tokens_out": 0, **base}
    t0 = time.perf_counter()
    try:
        r = call_llm(job["prompt"], max_tokens=MAX_TOKENS)
    except Exception as e:
        return {"output": rover, "status": f"error: {e}", "llm_seconds": round(time.perf_counter() - t0, 2),
                "calls": 1, "tokens_in": 0, "tokens_out": 0, **base}
    fills = parse_fills(r["text"], n)
    if r["cut"]:
        status = "cut_off"
    elif not fills:
        status = "unparseable"
    elif len(fills) < n:
        status = "partly_filled"
    else:
        status = "ok"
    return {"output": assemble(parts, fills), "status": status, "llm_seconds": round(time.perf_counter() - t0, 2),
            "calls": 1, "tokens_in": r["tokens_in"], "tokens_out": r["tokens_out"],
            "blanks": n, "blanks_filled": len(fills), "raw": r["text"].strip()}


wall = run_jobs(jobs, work)


## 4. Scores

In [ ]:
from itertools import combinations

out = pd.read_csv(RESULTS / "corrections" / f"{NAME}.csv", keep_default_na=False)
have_ids = set.intersection(*[set(out[out.condition == c].clip_id) for c in CONDITIONS])
by_clip["reference_words"] = by_clip.reference_clean.str.split().str.len()
scored = by_clip.clip_id.isin(have_ids).values
scorable = (by_clip.reference_clean.str.len() > 0).values & scored
print(f"Clips with outputs for every condition: {scored.sum()} of {len(by_clip)}")


def edit_counts(ref, hyp):
    if not hyp:
        return 0, len(ref.split()), 0, len(ref.split()), len(ref), len(ref)
    w = jiwer.process_words(ref, hyp)
    c = jiwer.process_characters(ref, hyp)
    return (w.substitutions, w.deletions, w.insertions, w.hits + w.substitutions + w.deletions,
            c.substitutions + c.deletions + c.insertions, c.hits + c.substitutions + c.deletions)


def counts_of(hyps):
    return np.array([edit_counts(r, h) if ok else (0, 0, 0, 0, 0, 0)
                     for r, h, ok in zip(by_clip.reference_clean, hyps, scorable)])


VERSIONS = {}
for stem, name in RECOGNISERS.items():
    VERSIONS[f"{name} (baseline)"] = {
        "hyp": list(by_clip[f"{stem}_clean"]), "recogniser_min_col": [f"{stem}_time_s"], "llm": None}
for cond, spec in CONDITIONS.items():
    o = out[out.condition == cond].drop_duplicates("clip_id", keep="last").set_index("clip_id").reindex(by_clip.clip_id)
    hyp = [clean_urdu(x) if isinstance(x, str) else "" for x in o.output.fillna("")]
    VERSIONS[spec["label"]] = {"hyp": hyp, "recogniser_min_col": [f"{s}_time_s" for s in spec["inputs"]],
                               "llm": o.reset_index(drop=True)}
ROVER_LABEL = "ROVER voting (experiment 04)"
rover_path = ROOT / "experiments" / "04_fusion_rover" / "results" / "corrections" / "rover.csv"
if rover_path.exists():
    ro = pd.read_csv(rover_path, keep_default_na=False).drop_duplicates("clip_id", keep="last").set_index("clip_id").reindex(by_clip.clip_id)
    VERSIONS[ROVER_LABEL] = {"hyp": [clean_urdu(x) if isinstance(x, str) else "" for x in ro.output.fillna("")],
                             "recogniser_min_col": [f"{s}_time_s" for s in RECOGNISERS], "llm": None}
else:
    print("ROVER results not found; run experiments/04_fusion_rover/rover.ipynb to add the ROVER comparison")
for v in VERSIONS.values():
    v["counts"] = counts_of(v["hyp"])

is_fleurs, is_us = by_clip.dataset.eq("FLEURS"), by_clip.dataset.eq("UrduSpeech")
SUBSETS = [
    ("FLEURS", "test + validation", is_fleurs & by_clip.subset.isin(["test", "validation"])),
    ("FLEURS", "train", is_fleurs & by_clip.subset.eq("train")),
    ("FLEURS", "test + validation + train", is_fleurs),
    ("UrduSpeech", "benchmark (human-checked references)", is_us & by_clip.subset.eq("benchmark")),
    ("UrduSpeech", "corpus (Gemini references)", is_us & by_clip.subset.eq("corpus")),
    ("UrduSpeech", "benchmark + corpus", is_us),
]
FILTERS = [("all clips", scorable), ("references with 3+ words", scorable & (by_clip.reference_words >= 3).values)]

rows = []
for dataset, subset, in_subset in SUBSETS:
    for filter_name, keep in FILTERS:
        m = in_subset.values & keep
        if not m.any():
            continue
        for label, v in VERSIONS.items():
            c = v["counts"][m]
            words = c[:, 3].sum()
            row = {"dataset": dataset, "subset": subset, "clips_scored": filter_name, "clips": int(m.sum()),
                   "reference_words": int(words), "version": label,
                   "WER_%": round(100 * c[:, :3].sum() / words, 2), "CER_%": round(100 * c[:, 4].sum() / c[:, 5].sum(), 2),
                   "substitutions_%": round(100 * c[:, 0].sum() / words, 2),
                   "deletions_%": round(100 * c[:, 1].sum() / words, 2),
                   "insertions_%": round(100 * c[:, 2].sum() / words, 2),
                   "recogniser_min": round(by_clip.loc[m, v["recogniser_min_col"]].astype(float).sum().sum() / 60, 1)}
            if v["llm"] is not None:
                l = v["llm"][m]
                row.update({"llm_min": round(l.llm_seconds.astype(float).sum() / 60, 1), "llm_calls": int(l.calls.astype(int).sum()),
                            "tokens_in": int(l.tokens_in.astype(int).sum()), "tokens_out": int(l.tokens_out.astype(int).sum()),
                            "errors_or_cut_off": int((l.status.astype(str).str.startswith("error") | l.status.eq("cut_off")).sum())})
            rows.append(row)
scores = pd.DataFrame(rows)
scores.to_csv(RESULTS / "scores" / f"{NAME}_scores.csv", index=False, encoding="utf-8-sig")


def paired_bootstrap(errors_a, errors_b, words, n=1000, seed=42):
    idx = np.random.default_rng(seed).integers(0, len(words), size=(n, len(words)))
    diff = 100 * (errors_a[idx].sum(1) - errors_b[idx].sum(1)) / words[idx].sum(1)
    return np.percentile(diff, [2.5, 97.5])


rows = []
for dataset, subset, in_subset in SUBSETS:
    for filter_name, keep in FILTERS:
        m = in_subset.values & keep
        if not m.any():
            continue
        for cond, spec in CONDITIONS.items():
            a = VERSIONS[spec["label"]]
            baselines = [f"{RECOGNISERS[stem]} (baseline)" for stem in spec["inputs"]]
            baselines += [ROVER_LABEL] if ROVER_LABEL in VERSIONS else []
            for b_label in baselines:
                b = VERSIONS[b_label]
                ea, eb = a["counts"][m, :3].sum(1), b["counts"][m, :3].sum(1)
                words = a["counts"][m, 3]
                lo, hi = paired_bootstrap(ea, eb, words)
                wa, wb = 100 * ea.sum() / words.sum(), 100 * eb.sum() / words.sum()
                changed = np.array([x != y for x, y in zip(np.array(a["hyp"], dtype=object)[m], np.array(b["hyp"], dtype=object)[m])])
                rows.append({"dataset": dataset, "subset": subset, "clips_scored": filter_name, "clips": int(m.sum()),
                             "corrected": spec["label"], "baseline": b_label,
                             "WER_corrected_%": round(wa, 2), "WER_baseline_%": round(wb, 2),
                             "difference": round(wa - wb, 2), "interval_low": round(lo, 2), "interval_high": round(hi, 2),
                             "result": "corrected lower" if hi < 0 else "baseline lower" if lo > 0 else "no reliable difference",
                             "lines_changed_%": round(100 * changed.mean(), 1),
                             "lines_improved": int(((ea < eb) & changed).sum()), "lines_worsened": int(((ea > eb) & changed).sum())})
vs = pd.DataFrame(rows)
vs.to_csv(RESULTS / "scores" / f"{NAME}_vs_baseline.csv", index=False, encoding="utf-8-sig")

shown = scores[scores.clips_scored == "all clips"].pivot_table(index=["dataset", "subset"], columns="version", values="WER_%", sort=False)
print("WER % on all clips")
print(shown.to_string())
print("\nDifference to the baseline recogniser (corrected minus baseline, 95% interval), all clips")
print(vs[vs.clips_scored == "all clips"][["subset", "corrected", "baseline", "difference", "interval_low", "interval_high", "result"]].to_string(index=False))

llm_rows = out[out.condition == "disagreement_v2"].drop_duplicates("clip_id", keep="last")
print("\nWhere each clip ended (status)")
print(llm_rows.status.value_counts().to_string())
asked = llm_rows[llm_rows.blanks.astype(int) > 0]
print(f"\nBlanks answered by the LLM: {asked.blanks_filled.astype(int).sum()} of {asked.blanks.astype(int).sum()} "
      "(the rest got the ROVER vote)")


## 5. Run information

In [ ]:
info_path = RESULTS / "run_info" / f"{NAME}.json"
previous = json.loads(info_path.read_text()) if info_path.exists() else {}
info = {
    "notebook": NAME,
    "llm": LLM_NAME,
    "workers": WORKERS,
    "prompt": "disagreement_fill_few_shot",
    "unanswered_blank": "ROVER vote for that blank",
    "temperature": 0.0, "max_tokens": MAX_TOKENS,
    "example_clips": list(examples.clip_id),
    "machine": {"python": platform.python_version(), "platform": platform.platform()},
    "packages": {p: importlib.metadata.version(p) for p in ("pandas", "numpy", "jiwer")},
    "normaliser": "clean_urdu v1",
    "bootstrap": {"resamples": 1000, "seed": 42},
    "llm_wall_minutes": round(previous.get("llm_wall_minutes", 0) + wall / 60, 1),
}
info_path.write_text(json.dumps(info, indent=2, ensure_ascii=False))
print(json.dumps(info, indent=2, ensure_ascii=False))